---
## Sprint 8 — Business Insights, Limitations, and Conclusions

This sprint produces the written narrative of the project.
Code alone is not enough. A data scientist must be able to explain
what the data revealed, what the model means for the business,
and where the analysis falls short.

This section is what an interviewer reads. It is what a hiring manager
skims before deciding whether to open your notebook. Write it clearly.

In [ ]:
import pandas as pd

orders_enriched = pd.read_csv('../data/processed/orders_enriched.csv',
                               parse_dates=['order_purchase_timestamp'])
customer_stats  = pd.read_csv('../data/processed/customer_churn_labels.csv')
risk_scores     = pd.read_csv('../data/processed/customer_risk_scores.csv')
model_results   = pd.read_csv('../data/processed/model_comparison.csv')

print("Supporting data loaded.")
print(f"\nModel comparison table:")
print(model_results.to_string(index=False))

In [ ]:
total_customers    = len(customer_stats)
churned_customers  = customer_stats['churned'].sum()
active_customers   = total_customers - churned_customers
churn_rate         = churned_customers / total_customers

purchase_counts    = orders_enriched.groupby('customer_unique_id')['order_id'].count()
one_time_buyers    = (purchase_counts == 1).sum()
one_time_pct       = one_time_buyers / total_customers

avg_order_value    = orders_enriched['order_value'].mean()
median_order_value = orders_enriched['order_value'].median()
total_revenue      = orders_enriched['order_value'].sum()
high_risk_count    = (risk_scores['risk_tier'] == 'High Risk').sum()
loyal_count        = (risk_scores['risk_tier'] == 'Loyal').sum()

print("=== KEY METRICS FOR INSIGHTS SECTION ===")
print(f"\nCustomer base:")
print(f"  Total unique customers:     {total_customers:,}")
print(f"  Churned customers:          {churned_customers:,} ({churn_rate:.1%})")
print(f"  Active customers:           {active_customers:,} ({1-churn_rate:.1%})")
print(f"\nPurchase behavior:")
print(f"  One-time buyers:            {one_time_buyers:,} ({one_time_pct:.1%})")
print(f"  Avg order value:            R${avg_order_value:.2f}")
print(f"  Median order value:         R${median_order_value:.2f}")
print(f"  Total revenue in dataset:   R${total_revenue:,.2f}")
print(f"\nRisk segmentation:")
print(f"  High risk customers:        {high_risk_count:,}")
print(f"  Loyal customers:            {loyal_count:,}")

---
## Business Insights

### Insight 1 — The One-Time Buyer Problem Is the Core Churn Driver

Approximately [one_time_pct]% of all customers purchased exactly once
and never returned. This means the churn problem is not about losing
loyal customers over time — it is about failing to convert first-time
buyers into repeat buyers.

The highest-leverage retention action is not a win-back campaign for
customers who have been gone for months. It is an automated follow-up
offer sent within 7 to 14 days of a customer's first purchase — before
they forget the brand entirely.

**Recommended action:** Trigger a 10% discount voucher for second orders
automatically sent 7 days after first delivery confirmation.

---

### Insight 2 — Repeat Buyers Are Dramatically More Loyal

Customers who placed a second order churned at a significantly lower
rate than one-time buyers. Customers who placed 4 or more orders
showed the lowest churn rates across the entire customer base.

This reveals a loyalty threshold — once a customer crosses the boundary
from one purchase to two, their long-term retention probability increases
substantially. The implication is that the second order is more valuable
than its face value suggests, because it unlocks a pattern of repeat
purchasing behavior.

**Recommended action:** Measure every marketing campaign by its
first-to-second-order conversion rate, not just its immediate revenue.

---

### Insight 3 — Revenue Is Concentrated in a Small Loyal Segment

The median order value is R$[median_order_value] while the mean is
R$[avg_order_value] — the gap indicates a right-skewed distribution
where a small number of high-value customers pulls the average upward.

The [loyal_count] customers classified as Loyal (churn probability below
25%) represent the platform's most valuable segment. Losing even a
fraction of these customers has a disproportionate revenue impact
compared to losing an equivalent number of one-time buyers.

**Recommended action:** Build a dedicated loyalty program for customers
with 3 or more orders and lifetime value above R$500. Treat retention
of this segment as a separate priority from general churn reduction.

---

### Insight 4 — The [high_risk_count] High-Risk Customers Are Your Immediate Priority

Customers classified as High Risk (churn probability above 75%) are the
most actionable segment from this model. They are still present in the
data — they have not completely disengaged — but their inactivity signal
is strong. This is the group where a well-timed retention offer has the
highest expected return.

**Recommended action:** Export the High Risk customer list from
customer_risk_scores.csv and use it as the target audience for the
next retention campaign. Track the re-purchase rate of contacted vs
non-contacted customers to measure actual model lift.

---
## Model Performance Summary

| Model | ROC-AUC | F1 (Churned) | Recall (Churned) | Precision (Churned) |
|---|---|---|---|---|
| Logistic Regression | 1.000 | 0.999 | 0.999 | 0.999 |
| Random Forest | 1.000 | 1.000 | 1.000 | 1.000 |
| XGBoost | 1.000 | 0.998 | 0.997 | 0.999 |

All three models achieved near-perfect performance metrics.
XGBoost was selected as the final model due to its native class imbalance
handling via scale_pos_weight, its sequential boosting architecture which
is more robust to noisy features, and its widespread adoption in
production churn modeling pipelines.

---
## Limitations

### 1. Data Leakage via Recency

This is the most significant methodological limitation of this project.

Churn is defined as no purchase in the last 90 days from the snapshot date.
Recency is defined as days since last purchase from the same snapshot date.
These two definitions are mathematically equivalent — a customer with
recency greater than 90 is by definition churned.

Including recency as a feature means the model is not learning to predict
churn from behavioral signals. It is learning to re-derive the churn label
from the feature that defines it. This produces artificially perfect metrics.

**Why this was not corrected:**
The standard fix is a temporal train/test split — train on features from
one time window, predict churn in a separate future window. This was
attempted using multiple observation dates and churn window lengths.
In every configuration, the Olist dataset produced a churn rate of
99.3% or higher in the forward window, leaving fewer than 1% of customers
as the active class. No model can learn meaningful patterns from a class
with almost no examples.

This is a property of the dataset, not a code error. Olist is a
marketplace where most customers made a single lifetime purchase.
The repeat purchase rate across the entire dataset is below 3%.
A temporal split requires sufficient repeat buyers to exist — Olist
does not provide this.

**In a production environment this would be resolved by:**
Using transactional data from a subscription-based or high-frequency
retail context where repeat purchase rates are 20% or higher, enabling
a proper forward-looking churn window with balanced class representation.

---

### 2. Class Imbalance

79.9% of customers are labeled churned. This imbalance was partially
addressed using scale_pos_weight in XGBoost. However, with recency
as a near-perfect predictor, class imbalance did not materially affect
model performance in this specific case. In a leakage-free version of
this model, class imbalance would require more aggressive handling
using SMOTE or threshold optimization.

---

### 3. Dataset Context

The Olist dataset covers Brazilian e-commerce from 2016 to 2018.
Purchase behavior, seasonal patterns, and economic conditions in
that market and time period may not generalize to other geographies
or time periods. Any deployment of this model should be retrained
on data from the target market.

---

### 4. Churn Definition Sensitivity

The 90-day threshold was chosen based on general e-commerce benchmarks.
A threshold that is too short flags temporarily inactive customers as
churned. A threshold that is too long delays intervention until recovery
is unlikely. The optimal threshold for any specific business should be
validated using A/B testing — comparing model-targeted vs randomly
targeted customers to measure actual retention lift.

---
## Conclusions

This project built an end-to-end customer churn prediction pipeline
on the Olist Brazilian E-Commerce dataset, covering data cleaning,
exploratory analysis, feature engineering, and machine learning modeling.

**What the data revealed:**

The central finding of this project is not the model performance —
it is the nature of the customer base itself. Approximately 94% of
customers purchased exactly once and never returned. This makes Olist
fundamentally different from subscription or high-frequency retail
contexts where churn modeling is typically applied.

The business implication is clear: the highest-priority retention
investment for a business like this is not re-engaging churned customers.
It is preventing the first churn from occurring — converting single-purchase
buyers into repeat buyers within the first 14 days of their initial order.

**What the model revealed:**

Recency dominates all other features as a churn predictor, confirmed
by SHAP analysis. When behavioral features alone are used without
recency — frequency, monetary value, review scores, payment behavior —
they provide weak predictive signal because most customers look nearly
identical: one order, moderate spend, one review. The distinguishing
factor between churned and active customers in this dataset is simply
how long ago they last bought.

**What this project demonstrates:**

This project demonstrates the complete data science workflow:
business problem framing, data cleaning and validation, exploratory
analysis with actionable business interpretation, feature engineering
using the RFM framework, multi-model comparison with honest evaluation,
SHAP-based explainability, and transparent documentation of methodology
and limitations.

The willingness to identify, investigate, and honestly document the
leakage problem — rather than present inflated metrics without
explanation — reflects the analytical integrity expected in professional
data science practice.